In [1]:
import sys
from pathlib import Path
from utils import make_figure, save_fig
current_dir = Path().resolve()
sys.path.append(current_dir.parent.parent.as_posix())
import os
import pandas as pd
from sonogenetics.analysis.lib.data_io import DataIO
import numpy as np
from sonogenetics.project_colors import ProjectColors
from sonogenetics.analysis.lib.analysis_params import dataset_dir, figure_dir_analysis

# Load data
if not os.path.exists(figure_dir_analysis):
    os.makedirs(figure_dir_analysis)
    
data_io = DataIO(dataset_dir)
session_id = '2026-03-25 mouse c57 617 Mekano6 A'

figure_dir_analysis = figure_dir_analysis / session_id
print(session_id)
data_io.load_session(session_id, load_pickle=True, load_waveforms=False)
data_io.dump_as_pickle()

loadname = dataset_dir / f'{session_id}_cells.csv'
cells_df = pd.read_csv(loadname, header=[0, 1], index_col=0)
clrs = ProjectColors()

# Print available recording ids
print("Available recording ids:")
for rec_id in data_io.recording_ids:
    print(f"- {rec_id}")

2026-03-25 mouse c57 617 Mekano6 A
Loading pickled data (not from h5 file)
Available recording ids:
- rec_2_A_20260325_pa_intensity_test
- rec_3_A_20260325_pa_dmd_timing
- rec_4_A_20260325_dmd_full_field


In [2]:
# Check trigger times

# Select a protocol
df = data_io.burst_df.query('recording_name == "rec_3_A_20260325_pa_dmd_timing"')

# Select a random trial
tid = list(df.train_id.unique())[12]
print(df.laser_onset_delay.unique())

fig = make_figure(
    width=1, height=1,
    x_domains={1: [[0.1, 0.9]], 2: [[0.1, 0.9]]},
    y_domains={1: [[0.55, 0.95]], 2: [[0.05, 0.45]]},
)

print(f'laser onset - dmd onset (positive values means laser lags behind dmd)')

for (lsr_delay, dmd_delay, tid), df2 in df.groupby(['laser_onset_delay', 'dmd_onset_delay', 'train_id']):
    if pd.isna(lsr_delay) or pd.isna(dmd_delay):
        continue

    in_legend = False
    y = df2.laser_burst_onset - df2.dmd_burst_onset

    print(f'dmd delay {dmd_delay:.0f}, laser delay {lsr_delay:.0f} | measured delay: {np.mean(y[1:]):.2f} ms')
    x = np.arange(0, y.size, 1)

    if lsr_delay == 0:
        row = 1
        delay = dmd_delay
        if dmd_delay == 20:
            clr = 'green'
        elif dmd_delay == 40:
            clr = 'red'
        elif dmd_delay == 0:
            clr = 'black'
        else:
            raise ValueError(f'include: {dmd_delay}')
    elif dmd_delay == 0:
        row = 2
        delay = lsr_delay
        if lsr_delay == 20:
            clr='green'
        elif lsr_delay == 40:
            clr = 'red'
        else:
            raise ValueError(f'include: {lsr_delay}')

    else:
        raise ValueError('one should be zero')

    plot_pos = dict(row=row, col=1)


    fig.add_scatter(
        x=x, y=y,
        mode='markers+lines',
        showlegend=not in_legend,
        name=f'{delay:.0f}',
        marker=dict(color=clr, size=4),
        line=dict(width=3),
        **plot_pos
    )

    in_legend=True

fig.update_xaxes(
    title_text='burst i'
)
fig.update_yaxes(
    tickvals=np.arange(-100, 100, 10),
    title_text='laser onset - burst onset [ms]',
    showgrid=False,
)
save_fig(
    fig=fig, savename=figure_dir_analysis / 'misc' / 'onset_delays', display=True
)
# fig.show()

# for delay, df2 in df.groupby('laser_onset_delay'):
#     print(f'{delay}')
#     mean_delay = np.mean(df2.laser_burst_onset - df2.dmd_burst_onset)




[ 0. 40. 20.]
laser onset - dmd onset (positive values means laser lags behind dmd)
dmd delay 0, laser delay 0 | measured delay: 0.40 ms
dmd delay 0, laser delay 0 | measured delay: 0.42 ms
dmd delay 0, laser delay 0 | measured delay: 0.41 ms
dmd delay 20, laser delay 0 | measured delay: -19.49 ms
dmd delay 20, laser delay 0 | measured delay: -19.65 ms
dmd delay 20, laser delay 0 | measured delay: -19.52 ms
dmd delay 40, laser delay 0 | measured delay: -39.67 ms
dmd delay 40, laser delay 0 | measured delay: -39.53 ms
dmd delay 40, laser delay 0 | measured delay: -39.53 ms
dmd delay 0, laser delay 20 | measured delay: 20.55 ms
dmd delay 0, laser delay 20 | measured delay: 20.20 ms
dmd delay 0, laser delay 20 | measured delay: 20.37 ms
dmd delay 0, laser delay 40 | measured delay: 40.51 ms
dmd delay 0, laser delay 40 | measured delay: 40.68 ms
dmd delay 0, laser delay 40 | measured delay: 40.70 ms
saved: E:\sono\figures\2026-03-25 mouse c57 617 Mekano6 A\misc\onset_delays.png
displaying 

In [11]:
cid = 'uid_2026-03-25 mouse c57 617 Mekano6 A_004'
ec = 216
rec_name = 'rec_2_A_20260325_pa_intensity_test'

df = data_io.burst_df.query(
    f'electrode == {ec} and '
    f'rec_id == "{rec_name}"'
)

# for tid, tdf in df.groupby('train_id'):
#     print(tid, f'{tdf.iloc[0].laser_power:.0f}, {tdf.iloc[0].laser_pulse_repetition_rate:.0f}')
#     print(np.diff(tdf.laser_burst_onset))
df.shape

(270, 50)

In [16]:
uid = r'uid_2026-03-25 mouse c57 617 Mekano6 A_004'
print(data_io.cluster_df.loc[uid, 'ch'])

173


In [17]:
from sonogenetics.analysis.dev_mua_analysis import read_triggers, get_filtered_daa_and_spikes

from sonogenetics.preprocessing.params import (data_sample_rate, data_type, data_nb_channels,
                                               data_trigger_channels)

filename = r"E:\sono\2026-03-25 mouse c57 617 Mekano6 A\raw\rec_2_A_20260325_pa_intensity_test.raw"


# Load channel data
trigger_channel = 'laser'
data = np.memmap(filename, dtype=data_type)
filename = Path(filename)
n_samples = int(data.size / data_nb_channels)
rec_duration = (n_samples / data_sample_rate) / 60  # [min]


# Define indices of current channel in data object
triggers = read_triggers(data, trigger_channel)
n_trains = len(triggers['train_onsets'])
n_bursts = len(triggers['burst_onsets'])

# Print file information
print(f'\t\t\t{filename.name}\n')
print(f'\trecording duration: {rec_duration:.0f} min\n')
print(f'\tdetected {n_trains} trains on {trigger_channel} trigger')
print(f'\tdetected {n_bursts} bursts on {trigger_channel} trigger')

print(tid)


reading chunks: 100%|██████████| 200/200 [00:04<00:00, 43.46it/s]

			rec_2_A_20260325_pa_intensity_test.raw

	recording duration: 33 min

	detected 27 trains on laser trigger
	detected 810 bursts on laser trigger
                                                                        rec_id  \
tid_2026-03-25 mouse c57 617 Mekano6 A_032  rec_2_A_20260325_pa_intensity_test   
tid_2026-03-25 mouse c57 617 Mekano6 A_044  rec_2_A_20260325_pa_intensity_test   
tid_2026-03-25 mouse c57 617 Mekano6 A_049  rec_2_A_20260325_pa_intensity_test   

                                            dmd_train_onset  \
tid_2026-03-25 mouse c57 617 Mekano6 A_032              0.0   
tid_2026-03-25 mouse c57 617 Mekano6 A_044              0.0   
tid_2026-03-25 mouse c57 617 Mekano6 A_049              0.0   

                                            laser_train_onset  \
tid_2026-03-25 mouse c57 617 Mekano6 A_032        186861.2500   
tid_2026-03-25 mouse c57 617 Mekano6 A_044        591653.8125   
tid_2026-03-25 mouse c57 617 Mekano6 A_049        758582.5625   

          

In [29]:
uid = r'uid_2026-03-25 mouse c57 617 Mekano6 A_004'
unit_ch = data_io.cluster_df.loc[uid, 'ch']
tid = data_io.train_df.query(
    f'laser_power == 6000 and '
    f'laser_pulse_repetition_rate == 6000 and '
    f'electrode == 216'
).iloc[0].train_id

print(tid, unit_ch)

bo = data_io.burst_df.query(f'train_id == "{tid}"')
print(bo.laser_burst_onset.max())



tid_2026-03-25 mouse c57 617 Mekano6 A_049 173
787457.4375


In [15]:
train_onsets = triggers['train_onsets']
burst_onsets = triggers['burst_onsets']
idx = np.where(
    (burst_onsets >= train_onsets[0]) &
    (burst_onsets < train_onsets[1])
)[0]
trigger_type = 'laser'
d = []
for ti, t in enumerate(burst_onsets[idx]):
    d.append(get_filtered_daa_and_spikes(data, data_trigger_channels[trigger_type], t, ti))


ValueError: The length of the input vector x must be greater than padlen, which is 21.

In [ ]:
idx = np.where()